# Assignment 3: Particle Charge Deposition and a Poisson Field Solver

**PHYS690: Computational Methods for Physics Research**  
**Unit 3: Boundary-value problems, sparse matrices, and grid-based field solvers**

**Student name(s):** _replace this text with your name(s)_  
**Private GitHub repository URL:** _replace this text with your assignment repository link_

## Purpose

This assignment combines the numerical-validation workflow from Lecture 10, the finite-difference Poisson solver from Lecture 11, and the source-to-field pipeline from Lecture 12. You will deposit a prescribed set of charged macroparticles onto a grid, solve Poisson's equation for the electrostatic potential, compute the electric field, and study how grid spacing and charge-assignment scheme change the result.

This is not a full PIC simulation: particles are not pushed in time, and there is no current deposition or Maxwell update. Instead, the assignment isolates the particle-to-grid and field-solve steps that appear inside larger PIC codes.

## Physics and computational setting

Production PIC codes represent plasma species with macroparticles moving through a spatial grid. The charge and/or current carried by those particles is deposited onto the grid, and the grid fields are updated from Maxwell's equations or from an electrostatic Poisson solve. WarpX describes this loop as field interpolation to particles, particle update, charge/current deposition to the grid, and field update from the grid source terms. Smilei is an open-source PIC code used for plasma simulation across laser-plasma and astrophysical applications. The eduPIC paper describes a one-dimensional electrostatic PIC/MCC teaching code for radio-frequency plasma simulation.

Your controlled calculation focuses on the electrostatic part:

$$-\nabla^2 V = \rho/\epsilon_0, \qquad \mathbf E = -\nabla V.$$

The charge density $\rho$ will come from prescribed particles. Grounded conducting walls are represented by Dirichlet boundary conditions, $V=0$, on the edge of the square domain.

## Deliverable

Submit your private GitHub repository containing a reproducible notebook and command-line script. Your repository should include:

- this completed notebook;
- a completed command-line script at `scripts/make_pic_poisson_comparison.py`;
- final figures saved in `figures/`;
- a concise summary table saved by the script, such as `figures/pic_poisson_summary.csv`;
- a `requirements.txt` file listing the Python packages needed to rerun the notebook and script;
- an updated `README.md` explaining how to rerun the work;
- a Git history showing meaningful progress through the assignment.

## Part 1: Set up the Python environment

Create and activate a local environment, then install the requirements.

On macOS:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

On Windows Git Bash:

```bash
python -m venv .venv
source .venv/Scripts/activate
python -m pip install -r requirements.txt
```

After that, select the `.venv` Python kernel for this notebook.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve

plt.rcParams.update({
    "figure.figsize": (7, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"SciPy: {scipy.__version__}")

## Part 2: Inspect a prescribed macroparticle set

A full PIC code may initialize particles from a distribution function, import particles from a restart file, or create particles through ionization/collision models. Here everyone will use the same prescribed particle set from Lecture 12 so that numerical results can be reproduced and compared.

The square domain is $0\le x\le 1$, $0\le y\le 1$. The particles form separated positive and negative clusters and remain more than one cell width from the boundary on the coarsest required grid. This matters because the grounded Poisson solve uses charge density only on interior grid nodes.

In [ ]:
LX = 1.0
LY = 1.0
EPS0 = 1.0

# Prescribed Lecture 12 macroparticles. Do not change these positions or charges:
# holding the physical source fixed is essential for a controlled comparison.
particles = pd.DataFrame({
    "x": [0.29, 0.33, 0.36, 0.39, 0.31, 0.61, 0.64, 0.67, 0.70, 0.66],
    "y": [0.46, 0.52, 0.48, 0.55, 0.58, 0.44, 0.50, 0.56, 0.47, 0.60],
    "q": [1.0, 0.8, 1.1, 0.9, 1.2, -1.0, -0.9, -1.1, -0.8, -1.2],
})

particles

In [ ]:
# TODO: Plot the prescribed particles.
#
# Complete instructions:
# 1. Use color to show the sign and magnitude of q.
# 2. Use equal aspect ratio so the square domain is not distorted.
# 3. Label the axes.
# 4. Add a colorbar.
fig, ax = ...
...
plt.show()

### Interpretation prompt

In a markdown cell below this one, describe the prescribed particle configuration. What plasma or beam situation might it evoke? What important physics is missing because the particles are prescribed rather than evolved? Also report the net particle charge $\sum_p q_p$.

_Your answer here._

## Part 3: Deposit particle charge on a grid

Charge deposition converts particles into a grid source term. Implement two schemes:

- **Nearest-grid-point (NGP):** deposit each particle charge to the nearest grid node.
- **Cloud-in-cell (CIC):** distribute each particle charge to the four surrounding grid nodes with bilinear weights.

In both cases, divide by the cell area $\Delta x\Delta y$ to convert charge to charge density. Track both total deposited charge and charge deposited on boundary nodes. The latter must be zero here: the grounded solver treats boundary values of $V$ as prescribed and only puts interior values of $\rho$ into its right-hand side.

In [ ]:
def deposit_ngp(particles, nx, ny, Lx=LX, Ly=LY):
    """Deposit particle charge to nearest grid point and return rho[j, i]."""
    # TODO: Implement nearest-grid-point deposition.
    #
    # Complete instructions:
    # 1. Compute dx = Lx / (nx - 1) and dy = Ly / (ny - 1).
    # 2. Create rho as a zeros array with shape (ny, nx).
    # 3. For each particle, convert x and y to nearest integer grid indices.
    # 4. Clip indices so roundoff cannot put a particle outside the grid.
    # 5. Add q / (dx * dy) to rho[j, i].
    #
    # Course reference: Lecture 11 uses j, i array indexing for grid values.
    ...


def deposit_cic(particles, nx, ny, Lx=LX, Ly=LY):
    """Deposit particle charge with cloud-in-cell weighting and return rho[j, i]."""
    # TODO: Implement bilinear cloud-in-cell deposition.
    #
    # Complete instructions:
    # 1. Compute fractional grid coordinates gx = x / dx and gy = y / dy.
    # 2. Let i0=floor(gx), j0=floor(gy), and define fractional offsets
    #    wx = gx - i0 and wy = gy - j0.
    # 3. Deposit to the four nodes (i0,j0), (i0+1,j0), (i0,j0+1),
    #    (i0+1,j0+1) with weights:
    #       (1-wx)(1-wy), wx(1-wy), (1-wx)wy, wx wy.
    # 4. Divide charge by dx * dy to make charge density.
    # 5. Use particles away from the boundary so i0+1 and j0+1 are valid.
    #
    # Reference: WarpX and Smilei both describe particle quantities being
    # deposited from macroparticles to a grid; this is the simplest version.
    ...

In [ ]:
# TODO: Compare deposited charge density for NGP and CIC on the same grid.
#
# Complete instructions:
# 1. Choose nx = ny = 33.
# 2. Compute rho_ngp and rho_cic.
# 3. Print the total particle charge and the integrated deposited charge
#    sum(rho) * dx * dy for each scheme.
# 4. Construct a Boolean mask that selects all boundary nodes. Print the
#    integrated boundary-node charge for each scheme and verify it is zero.
#    A nonzero value would be omitted from this solver's source vector b.
# 5. Plot both charge-density arrays with imshow using the same color scale.
nx = ny = 33
...

### Interpretation prompt

In a markdown cell below this one, compare NGP and CIC. Which charge density looks smoother? Does each scheme conserve total deposited charge to roundoff? Is any charge deposited on boundary nodes? Explain why both checks matter for this field solve.

_Your answer here._

## Part 4: Assemble and solve Poisson's equation

Use the Lecture 11 finite-difference pattern for

$$-\nabla^2 V = \rho/\epsilon_0$$

with grounded Dirichlet boundaries, $V=0$ on all four edges. Only interior grid values are unknown. The five-point stencil becomes a sparse matrix. Because the boundary values are zero, there are no nonzero boundary contributions to the right-hand side for this assignment.

In [ ]:
def interior_index(i, j, nx):
    """Map interior grid index (i, j) to a 1D unknown-vector index."""
    return (j - 1) * (nx - 2) + (i - 1)


def build_poisson_system(rho, Lx=LX, Ly=LY, eps0=EPS0):
    """Build sparse A and b for -laplacian(V) = rho / eps0."""
    # TODO: Assemble A and b.
    #
    # Complete instructions:
    # 1. Read ny, nx from rho.shape.
    # 2. Compute dx, dy.
    # 3. There are (nx-2)*(ny-2) unknown interior grid points.
    # 4. Use lil_matrix for assembly, then convert to CSR before returning.
    # 5. For each interior point, add the diagonal coefficient
    #       2/dx**2 + 2/dy**2.
    # 6. Add off-diagonal neighbor coefficients -1/dx**2 or -1/dy**2
    #    when the neighbor is also an interior unknown.
    # 7. Set b[row] = rho[j, i] / eps0.
    #
    # Course reference: Lecture 11 Part 3, build_dirichlet_poisson_system.
    ...


def solve_potential(rho, Lx=LX, Ly=LY, eps0=EPS0):
    """Solve for V on the full grid including grounded boundaries."""
    # TODO: Solve the sparse linear system.
    #
    # Complete instructions:
    # 1. Call build_poisson_system.
    # 2. Solve with scipy.sparse.linalg.spsolve.
    # 3. Create a zeros array V with the same shape as rho.
    # 4. Unpack the interior solution vector into V[1:-1, 1:-1].
    # 5. Return V.
    ...

### Part 4A: Validate the solver with a manufactured solution

Before applying the solver to grid-scale particle sources, validate it with a smooth problem whose answer is known. Use

$$
V_{\rm exact}(x,y)=\sin\left(\frac{\pi x}{L_x}\right)
\sin\left(\frac{\pi y}{L_y}\right),
$$

which satisfies the grounded boundaries. Substitution into $-\nabla^2V=\rho/\epsilon_0$ gives

$$
\rho(x,y)=\epsilon_0\pi^2\left(\frac{1}{L_x^2}+\frac{1}{L_y^2}\right)V_{\rm exact}(x,y).
$$

This separates verification of the finite-difference solver from the later sensitivity study with nonsmooth deposited particles. It follows the convergence workflow used in Lectures 10 and 11.

In [ ]:
# TODO: Measure the grid convergence of solve_potential for the smooth benchmark.
#
# Complete instructions:
# 1. Loop over n = [17, 33, 65]. For each n, create x, y, X, and Y arrays.
# 2. Evaluate V_exact and the manufactured rho from the equations above.
# 3. Compute V_numerical = solve_potential(rho).
# 4. Record dx and max(abs(V_numerical - V_exact)) in a list of dictionaries.
# 5. Convert the rows to a DataFrame and display it.
# 6. Use np.polyfit(log(dx), log(max_error), 1) to estimate the observed order.
# 7. Make a log-log error-versus-dx plot and save it as
#    figures/pic_poisson_manufactured_convergence.png.
#
# References: Lecture 10's measured-order workflow and Lecture 11 Part 5.
validation_rows = []
...
validation_table = ...
validation_table

#### Validation interpretation

Report the observed order and compare it with the expected value of approximately 2 for a centered five-point Laplacian. Explain what this benchmark tests that the algebraic residual does not.

_Your answer here._

### Part 4B: Apply the solver to the deposited particle source

Now solve the controlled particle problem. Unlike the smooth manufactured benchmark, deposited point-particle sources become increasingly concentrated as the grid is refined, so near-particle maxima are not expected to show ordinary second-order convergence.

In [ ]:
# TODO: Solve Poisson's equation for one reference case.
#
# Complete instructions:
# 1. Use rho_cic from the previous section.
# 2. Solve for V_cic.
# 3. Check the algebraic residual norm ||A u - b|| for the interior solution.
# 4. Plot V_cic with imshow and a colorbar.
#
# Course reference: Lecture 11 distinguishes algebraic residual from
# discretization error. Here there is no analytic solution, so residual
# checks whether the linear system was solved accurately.
...

### Interpretation prompt

In a markdown cell below this one, explain the role of the boundary condition. What physical object does a grounded Dirichlet boundary represent? Why is the algebraic residual not the same thing as a complete error estimate?

_Your answer here._

## Part 5: Compute the electric field

The electric field is

$$\mathbf E = -\nabla V.$$

Compute $E_x$ and $E_y$ using finite differences. Then plot the potential and overlay field vectors or streamlines.

In [ ]:
def electric_field(V, Lx=LX, Ly=LY):
    '''Return Ex and Ey from V using finite differences.'''
    # TODO: Compute E = -grad(V).
    #
    # Complete instructions:
    # 1. Compute dx and dy from V.shape and domain size.
    # 2. Use np.gradient(V, dy, dx, edge_order=2) or explicit finite differences.
    #    Remember that axis 0 is y and axis 1 is x.
    # 3. Return Ex = -dV_dx and Ey = -dV_dy.
    ...


# TODO: Compute Ex and Ey for the reference potential and make a field plot.
# Save a polished version to figures/pic_poisson_potential_field.png.
...

### Interpretation prompt

In a markdown cell below this one, describe the field pattern. Where is the field strongest? How does the sign of the nearby charges affect the direction of the field?

_Your answer here._

## Part 6: Define diagnostics for comparison

Attach a diagnostic to each stage of the source-to-field pipeline, following Lecture 12. Include total deposited charge, boundary-node charge, the algebraic residual, a Gauss-law consistency error, and field observables. The field-energy-like integral is

$$U_E = \frac{\epsilon_0}{2}\int |\mathbf E|^2\,dA.$$

Also record $V$ and $|\mathbf E|$ at the fixed probe $(x,y)=(0.50,0.75)$, which lies away from the particle clusters and is represented exactly on both required grids. Values such as `max_abs_E` and field energy are useful sensitivity indicators, but for grid-scale particle sources they are dominated by the near-particle field and should not be labeled converged quantities.

In [ ]:
def field_summary(V, Ex, Ey, rho, Lx=LX, Ly=LY, eps0=EPS0):
    '''Return scalar diagnostics for one solved field.'''
    # TODO: Compute and return a dictionary with at least:
    # - total_charge = sum(rho) * dx * dy
    # - boundary_charge, integrating rho only over boundary nodes
    # - relative_residual: rebuild A and b, flatten V[1:-1, 1:-1] as u,
    #   and compute norm(A @ u - b) / max(norm(b), 1)
    # - max_abs_V and max_abs_E
    # - field_energy = 0.5 * eps0 * sum(Ex**2 + Ey**2) * dx * dy
    # - probe_V and probe_E at (x, y) = (0.50, 0.75)
    # - gauss_rms: the RMS of div(E) - rho/eps0 on [1:-1, 1:-1]
    #
    # Use np.gradient with the same [y, x] axis ordering as electric_field.
    # The Gauss-law error tests the full rho -> V -> E pipeline and will not be
    # as small as the algebraic residual, especially near grid-scale sources.
    ...


def run_case(particles, nx, ny, scheme):
    '''Deposit, solve, compute field, and summarize one case.'''
    # TODO: Use deposit_ngp or deposit_cic depending on scheme.
    # Then solve for V, compute Ex/Ey, and return rho, V, Ex, Ey, summary.
    ...

## Part 7: Resolution and charge-assignment sensitivity

Make a controlled comparison using the same prescribed particles for all cases. You are required to compare both NGP and CIC on both `33 x 33` and `65 x 65` grids. When comparing schemes, hold the grid fixed; when comparing grids, hold the scheme fixed.

This is a sensitivity study, not a conventional convergence test. A deposited particle represents a grid-scale approximation to a point source, so refining the grid makes its charge density and near-particle field increasingly concentrated. The smooth manufactured problem in Part 4A, rather than `max_abs_E` here, establishes the solver's formal convergence.

In [ ]:
# TODO: Run a small comparison study.
#
# Complete instructions:
# 1. Loop over grid sizes [33, 65].
# 2. Loop over schemes ["NGP", "CIC"].
# 3. Store outputs in a dictionary keyed by (nx, scheme).
# 4. Collect summary dictionaries into a pandas DataFrame.
# 5. Display the summary table.
outputs = {}
summary_rows = []
...
summary_table = ...
summary_table

In [ ]:
# TODO: Make the required sensitivity-comparison figure and save it.
#
# Complete instructions:
# 1. For all four cases, extract V along the fixed line y = 0.75.
#    This y value lies exactly on both required grids.
# 2. Plot V(x, y=0.75) for NGP and CIC on the 33 and 65 point grids.
# 3. Label every curve by grid size and deposition scheme.
# 4. Optionally add a second panel for probe_V, probe_E, or field_energy.
# 5. Save the figure to
#    figures/pic_poisson_grid_or_scheme_comparison.png.
#
# Do not describe agreement of max_abs_E or field_energy as formal grid
# convergence for these particle sources.
...

### Interpretation prompt

In a markdown cell below this one, explain what changed with grid spacing and deposition scheme. Compare the fixed probe and line-profile results with near-particle quantities such as `max_abs_E`. Which features appear robust, and which are sensitive to grid-scale source representation? State clearly what this comparison does and does not establish. Discuss the algebraic residual, manufactured-solution error, and Gauss-law error as distinct diagnostics.

_Your answer here._

## Part 8: Connect back to PIC codes and plasma physics

Write a short discussion connecting your controlled calculation to the PIC algorithm described by WarpX, Smilei, and eduPIC. Address:

- Which step of the PIC loop did your charge-deposition function represent?
- Which step did your Poisson solve represent?
- What would be needed to turn this static calculation into a time-dependent electrostatic PIC code?
- Why can charge-assignment schemes and grid spacing affect fields and therefore particle motion?
- What did this assignment omit compared with production plasma codes?

_Your discussion here._

## Part 9: Write a command-line reproduction script

A reproducible analysis should not depend only on notebook state. Complete `scripts/make_pic_poisson_comparison.py` so it reproduces the final outputs from the command line.

Your script should run from the repository root with:

```bash
python scripts/make_pic_poisson_comparison.py
```

The script must:

- use the same prescribed Lecture 12 particles;
- implement both NGP and CIC deposition;
- assemble and solve the same sparse Poisson system;
- reproduce the manufactured-solution validation;
- compute the electric field and all required diagnostics;
- run both schemes on the 33 and 65 point grids;
- save `figures/pic_poisson_potential_field.png`;
- save `figures/pic_poisson_manufactured_convergence.png`;
- save `figures/pic_poisson_grid_or_scheme_comparison.png`;
- save `figures/pic_poisson_summary.csv`.

## Part 10: Update README and submit

Update `README.md` so another person can rerun your work. Include setup instructions, the notebook name, the command to run the script, the names of generated figures, and a concise interpretation of your comparison.

Suggested final commands:

```bash
git status
git add README.md requirements.txt figures/*.png figures/*.csv scripts/make_pic_poisson_comparison.py notebooks/assignment3_pic_poisson_solver.ipynb
git commit -m "Complete PIC Poisson field-solver assignment"
git status
git log --oneline --max-count=5
git push -u origin submission
```

Finally, submit a pull request from the `submission` branch into `main` in your private assignment repository.

## References

- WarpX theory overview: https://warpx.readthedocs.io/en/latest/theory/intro.html
- Smilei PIC code documentation: https://smileipic.github.io/Smilei/
- Z. Donko et al., *eduPIC: an introductory particle based code for radio-frequency plasma simulation*, arXiv: https://arxiv.org/abs/2103.09642
- Course lectures: https://github.com/jrstevenjlab/gradcompphys/tree/main/lectures
- Lecture 10 IVP and numerical-validation notebook: https://github.com/jrstevenjlab/gradcompphys/blob/main/lectures/lecture10_9_29_26_ivp_ode.ipynb
- Lecture 11 finite-difference Poisson/BVP notebook: https://github.com/jrstevenjlab/gradcompphys/blob/main/lectures/lecture11_10_6_26_bvp_pde.ipynb
- Lecture 12 source-to-field and particle-deposition notebook: https://github.com/jrstevenjlab/gradcompphys/blob/main/lectures/lecture12_10_15_26.ipynb
- SciPy sparse linear algebra: https://docs.scipy.org/doc/scipy/reference/sparse.linalg.html